# Import you data
## Read raw data from file
Your raw data form the basis of all analyses. Keep the original dataset unchanged!


In [ ]:
df = read.csv("ihsm26_StudentLifeData.isle.csv", header = TRUE, stringsAsFactors = FALSE, fileEncoding = "UTF-8-BOM")
# check
nrow(df)

## Clean data

Use code to document and apply all data cleaning and transformations. This makes your work transparent and reproducible, even years after you have completed your analysis. 

Extend this code with even more cleaning steps if required. Carefully comment your code to document your decisions. 

In [ ]:
# replace all "" and "N/A" with NA to correctly define missing values 
df[df == "" | df == "N/A"] = NA
# check
sum(df == "" | df == "N/A", na.rm=T) # should be zero now (no "" left in df)

In [ ]:
# replace long value labels
df$SQ7[df$SQ7 == "STEM (Science. Technology. Engineering and Mathematics)"] = "STEM"
table(df$SQ7)

In [ ]:
# Initialize the cleaned variable; unclassifiable values remain NA
df$SQ3_4 = NA

# Austria
df$SQ3_4[df$SQ3 %in% c(
  "at", "AT", "austria", "Austria", "Austria Tirol",
  "Austria/Österreich", "Innsbruck", "österreich",
  "Österreich", "Salzburg", "Styria"
)] = "Austria"

# Germany
df$SQ3_4[df$SQ3 %in% c(
  "Deutschland", "Dresden", "German", "germany", "Germany"
)] = "Germany"

# EU/EEA countries other than Austria and Germany
df$SQ3_4[df$SQ3 %in% c(
  "belgium", "Belgium", "Bratislava", "Czechia", "Denmark",
  "finland", "Finland", "Finland erasmus", "France", "Genoa",
  "Hungary", "italia", "Italy", "lapland", "Magyarország", "Nederland",
  "Netherlands", "Netherlands/ Finland (exchange)",
  "Norway", "Spain", "Sweden", "The Netherlands"
)] = "EU/EEA"

# Countries outside the EU/EEA
df$SQ3_4[df$SQ3 %in% c(
  "America", "Australia", "Bangladesh", "Egypt", "england",
  "India", "Japan", "Mexico", "Mozambique", "Nepal",
  "Scotland", "South Korea", "Swistzerland", "Switzerland",
  "UK", "United Kingdom", "UNITED KINGDOM!!!!!!!",
  "united states", "United States", "US", "USA"
)] = "Other"

# Unclassifiable responses remain NA:
# "Political Science", "The world", "Wie"

# Display frequencies, including missing values
table(df$SQ3_4, useNA = "ifany")

## Define the final subsample

In [ ]:
# all respondents
table(df$SQ4)

In [ ]:
# exclude respondents who do not match the target population
df = df[df$SQ4 != "Secondary education" & !is.na(df$SQ4),]
# check
table(df$SQ4, useNA = "ifany")

In [ ]:
# Size of final sample
nrow(df)

# Analysis templates

In order to create generic templates, the following sample dataframe will be defined:

In [ ]:
# NOTE: This step is NOT required! The new sample dataframe "sdf" is defined for demonstration purposes only. 
# It contains two nominal scales (N1, N2), two ordinal scales (O1, O2), and a single metric scale (M1). 
# In your homework, simply refer to df (as imported from file in the very first cell above) and its respective variables 
sdf = data.frame(N1 = df$SQ3_4, N2 = df$SQ8, O1 = df$R3, O2 = df$DH3, M1 = df$SQ1)
variable_labels = c(N1 = "What is the country of your studies?",
                    N2 = "Are/were you working for pay to make your daily living while studying?",
                    O1 = "How would you rate your sleep quality overall?",
                    O2 = "How much do you trust digital health tools to provide reliable health information?",
                    M1 = "How old are you? (years)"
                  )
summary(sdf)


## Describe distributions

### Nominal Scales

In [ ]:
# nominal scales
t = table(sdf$N1, useNA="always")
t = data.frame(Value = names(t), Count = as.numeric(t), Percent = as.numeric(prop.table(t)))
t[c(1,3,2,4,5),]  # sort output as ranking if required

### Ordinal Scales

In [ ]:
# ordinal scales
t = table(sdf$O1, useNA="always")
t = data.frame(Value = names(t), Count = as.numeric(t), Percent = as.numeric(prop.table(t)))
t[c(5,2,3,1,4,6),] # adjust output to predefined order


### Metric Scales

In [ ]:
# use summary() to get the most relevant descriptive statistics in a single batch, ...
summary(sdf$M1)

In [ ]:
# ... extend the summary by additional parameters and reorder them to your taste, ...
c(summary(sdf$M1), SD = sd(sdf$M1, na.rm=T))[c(1, 2, 3, 5, 6, 4, 8, 7)]

In [ ]:
# ... or compile and format your individual selection 
round(c(quantile(sdf$M1, c(.33, .5, .67), na.rm=T), 
s = sd(sdf$M1, na.rm=T),
"s^2" = var(sdf$M1, na.rm=T)), 3)

## Crosstabulation (D1)

In [ ]:
table(sdf$N1, sdf$N2)

In [ ]:
# add totals
t = table(sdf$N1, sdf$N2)
t = cbind(t, Total = rowSums(t))
t = rbind(t, Total = colSums(t))
t

### Total percentages 


In [ ]:
# total percentages 
t = table(sdf$N1, sdf$N2)
t = prop.table(t)
t = cbind(t, Total = rowSums(t))
t = rbind(t, Total = colSums(t))
t*100

### Row percentages 


In [ ]:
# row percentages 
# how does N2 differ between groups of N1?
t = table(sdf$N1, sdf$N2)
t = rbind(t, Total = colSums(t))
# save the total counts for later use
n = rowSums(t)
t = prop.table(t, 1)
t = cbind(t, Total = rowSums(t))
# format as percentages
t = round(t*100, 1)
# we have to add the total counts for each 100% total to allow for evaluating the percentages correctly
t = cbind(t, N = n)
t

### Column percentages 


In [ ]:
# column percentages 
# how does N1 differ between groups of N2?
t = table(sdf$N1, sdf$N2)
t = cbind(t, Total = rowSums(t))
# save the total counts for later use
n = colSums(t)
t = prop.table(t, 2)
t = rbind(t, Total = colSums(t))
# format as percentages
t = round(t*100, 1)
# we have to add the total counts for each 100% total to allow for evaluating the percentages correctly
t = rbind(t, N = n)
t

## Group parameters (D2)


Use the functions `by()` or `tapply()` to provide grouped parametric statistics. In the following, `tapply()` is used because it provides simpler output. You may replace it with `by()` if you prefer the latter.

In [ ]:
# tapply(data, group, statistics)
tapply(sdf$M1, sdf$N1, mean)

In [ ]:
# As in the univariate case, you must add the na.rm flag in order to ignore NAs. Simply append any additional input.
tapply(sdf$M1, sdf$N1, mean, na.rm=T)

In [ ]:
# Note: to include quantiles use quantile(variable, collection of fractions between 0 and 1, na.rm=T)
quantile(sdf$M1, c(.25, .33, .5, .67, .75), na.rm=T)

In [ ]:
# use any parameter function of your choice and sort columns to your needs ...
tapply(sdf$M1, sdf$N1, median, na.rm=T)[c(1,3,2,4)] # Note: Indices define columns, no comma required

In [ ]:
# ... or compile your individual selection, ...
Min = tapply(sdf$M1, sdf$N1, min, na.rm=T)
Q1 = tapply(sdf$M1, sdf$N1, quantile, .25, na.rm=T)
Median = tapply(sdf$M1, sdf$N1, median, na.rm=T)
Q3 = tapply(sdf$M1, sdf$N1, quantile, .75, na.rm=T)
Max = tapply(sdf$M1, sdf$N1, max, na.rm=T)
Mean = tapply(sdf$M1, sdf$N1, mean, na.rm=T)
SD = tapply(sdf$M1, sdf$N1, sd, na.rm=T)
n = as.numeric(table(sdf$N1, useNA="no"))
cbind(Min, Q1, Median, Q3, Max, Mean, SD,n)[c(1,3,2,4),]  # Note the position of the comma: Indices define rows!


In [ ]:
# ... formatted to your taste
rbind(Min, Q1, Median, Q3, Max, Mean, SD, n)[,c(1,3,2,4)]  # Note the position of the comma: Indices define columns!


## Long scales as independent Variables (X)

When examining a metric independent variable in relation to a dependent variable with few response categories, grouping the metric variable into meaningful categories makes the results easier to present and interpret. In this example, the metric variable is divided into three approximately equal-sized groups using the 33rd and 67th percentiles as cut-off points. Use a different classification scheme if it is more appropriate.


In [ ]:
# compute and check class bounds
terciles = quantile(sdf$M1, c(.33, .67), na.rm=T)
terciles

In [ ]:
# apply class bounds for recoding M1 into a new variable M1_3 (read: "M1 recoded into 3 classes")
sdf$M1_3 = NA
sdf$M1_3[sdf$M1 <= terciles[1]] = "low"
sdf$M1_3[sdf$M1 > terciles[1] & sdf$M1 <= terciles[2]] = "medium"
sdf$M1_3[sdf$M1 > terciles[2]] = "high"
table(sdf$M1_3)

Now continue with M1_3 to build a crosstable with your dependent variable of choice 